In [1]:
%load_ext autoreload
%autoreload 2

# Automatic Fix AV classification

This notebook demonstrates how to fix a predicted AV classification map using topological recontrustion of the vascular tree.


In [2]:
from pathlib import Path

import numpy as np
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit.pipelines import AVSegToTree, GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import fix_av_map
from fundus_vessels_toolkit.utils.data_io import load_label_image
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

### Load a fundus image and its AV map


In [3]:
IMG = Path("g_005.png")
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/GAVE-train/")


# Path to the raw fundus image
RAW_PATH = PATH / "1-images" / IMG

# Path to the artery/vein segmentation
AV = PATH / "2-av" / IMG
TOPO = PATH / "3-topo" / IMG.with_suffix("")

fundus = FundusData(image=RAW_PATH, av=AV)
od_mac = segment(open_image(RAW_PATH)).numpy(force=True).argmax(axis=0)
fundus = fundus.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")

In [ ]:
from fundus_vessels_toolkit.vascular_data_objects.fundus_data import AVLabel
from fundus_vessels_toolkit.vascular_data_objects.vtree import VTree
from skimage.morphology import skeletonize as skimage_skeletonize


av2tree = NaiveAVSegToTree()
av2graph = AVSegToTree()

trees = VTree.load(str(TOPO) + "_art.npz"), VTree.load(str(TOPO) + "_vei.npz")
fundus_fixed = fundus.update(
    av=fix_av_map(fundus.av, trees, force_initial_segmentation=False, expand_labels_by=1, discard_av=False)
)
art_skel = skimage_skeletonize(fundus.artery_map)
vei_skel = skimage_skeletonize(fundus.vein_map)
fundus_skel = fundus.update(av=art_skel.astype(np.uint8) * 1 + vei_skel.astype(np.uint8) * 2)

m = Mosaic(3, cell_height=800)
fundus.draw(view=m[0])
draw_graph(av2graph.to_vgraph(fundus), m[0], branch_color=["grey"])
# draw_trees(av2tree(fundus), m[0], bspline_dir=True)
fundus_fixed.draw(view=m[1])
draw_trees(trees, m[1], bspline_dir=True)
fundus.draw(view=m[2])
m

GridBox(children=(View2D(linkedTransformGroup='5acec48759e94206bcc2d87ea23d4149'), View2D(linkedTransformGroup…

In [5]:
from fundus_vessels_toolkit.models.metrics.topological import (
    sample_valid_path_ratio,
    valid_path_ratio,
    valid_path_ratio_cpp,
)

fundus_art = fundus.av == 1
fundus_fixed_art = fundus_fixed.av == 1

(cor_art, n_art), (inf_art, n_fixed_art) = valid_path_ratio(fundus_art, fundus_fixed_art, skeleton=True)
(cor_vei, n_vei), (inf_vei, n_fixed_vei) = valid_path_ratio(fundus.av == 2, fundus_fixed.av == 2, skeleton=True)

%timeit valid_path_ratio(fundus_art, fundus_fixed_art, skeleton=True)
%timeit valid_path_ratio_cpp(fundus_art, fundus_fixed_art, skeleton=True)

print(f"Arteries: {cor_art:.1%} correct paths, {inf_art:.1%} incorrect paths")
print(f"Veins: {cor_vei:.1%} correct paths, {inf_vei:.1%} incorrect paths")

40.1 ms ± 121 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
38.3 ms ± 92.4 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Arteries: 93.3% correct paths, 98.8% incorrect paths
Veins: 97.0% correct paths, 68.2% incorrect paths


In [6]:
n_art, n_fixed_art

(3577009, 3675218)

In [7]:
valid_ratio_art = sample_valid_path_ratio(fundus_art, fundus_fixed_art, int(1e9))
valid_ratio_vei = sample_valid_path_ratio(fundus.av == 2, fundus_fixed.av == 2, int(1e9))
print(valid_ratio_art)
print(valid_ratio_vei)

((0.9875590801239014, 1.0), (0.9331189393997192, 0.9999997019767761))
((0.681696355342865, 1.0), (0.969709575176239, 0.9999995827674866))


In [ ]:
stats = []
for n, r in {1e2: 10, 1e3: 10, 1e4: 5, 1e5: 5, 1e6: 5, 1e7: 2, 1e8: 1}.items():
    for _ in range(r):
        valid_ratio_vei = sample_valid_path_ratio(fundus.av == 2, fundus_fixed.av == 2, int(n))
        stats.append(
            {
                "sample_size": n,
                "inf_": valid_ratio_vei[0][0],
                "inf_len": np.prod(valid_ratio_vei[0]).item(),
                "cor_": valid_ratio_vei[1][0],
                "cor_len": np.prod(valid_ratio_vei[1]).item(),
            }
        )

In [28]:
stats

[{'sample_size': 100.0,
  'valid_ratio_vei': ((0.6399999856948853, 1.0), (0.9799999594688416, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.8199999928474426, 1.0), (1.0, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.6200000047683716, 1.0), (0.9799999594688416, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.6399999856948853, 1.0), (1.0, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.5999999642372131, 1.0), (1.0, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.6399999856948853, 1.0), (0.9799999594688416, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.699999988079071, 1.0), (0.9799999594688416, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.5600000023841858, 1.0), (1.0, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.7400000095367432, 1.0), (0.9799999594688416, 1.0))},
 {'sample_size': 100.0,
  'valid_ratio_vei': ((0.6200000047683716, 1.0), (0.9399999976158142, 1.0))},
 {'sample_size': 1000.0,
  'valid_ratio_v

In [27]:
import plotly.express as px

px.scatter(stats, x="sample_size", y="valid_ratio_vei", log_x=True)

In [8]:
STOP

NameError: name 'STOP' is not defined

## Parse tree on the GT


Parse the topology of the ground truth segmentation and generate its topology map.


In [ ]:
seg2tree = NaiveAVSegToTree()
trees_gt = seg2tree(fundus_gt)

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

- The `labels` indicate to which subtree each vessel pixel belong, as well as the branching patterns to reach it.
- The `topo` map monotonically increases with the distance from the subtree root.


In [ ]:
m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=400,
)
fundus_gt.draw(view=m[0, 0])
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=True)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=True)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

## Parse graph on the Prediction


In [ ]:
trees_fixed = AVSegToTree()(fundus)

fundus_fixed = fundus.update(vessels=fix_av_map(fundus.av, trees_fixed))

m = Mosaic(2, cols_titles=["Predicted", "Corrected"], cell_height=800)
fundus.draw(view=m[0])
fundus_fixed.draw(view=m[1])

draw_trees(trees_fixed, view=m[1])
m